In [4]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import ollama
import json
from typing import List, Dict

In [5]:
# Analyze Page Function
async def analyze_page(url: str) -> Dict:
    """
    Opens the URL with Playwright and extracts page information
    Returns structured data about the page
    """
    async with async_playwright() as p:
        # Launch browser
        browser = await p.chromium.launch(headless=False)  # isko True krna hai baadme prod me
        page = await browser.new_page()
        
        await page.goto(url, wait_until="networkidle")
        
        # waiting for some dynamic content or slow loading
        await page.wait_for_timeout(2000)
        
        # get page title
        title = await page.title()
        
        # Get page HTML
        html_content = await page.content()
        
        # take screenshot
        screenshot_path = "page_screenshot.png"
        await page.screenshot(path=screenshot_path, full_page=True)
        
        # for parse HTML with
        soup = BeautifulSoup(html_content, 'html.parser')
        
        # extract key elements
        page_info = {
            "url": url,
            "title": title,
            "headings": [],
            "links": [],
            "forms": [],
            "buttons": [],
            "inputs": [],
            "images": [],
            "text_content": ""
        }
        
        # extract headings (h1, h2, h3)
        for tag in ['h1', 'h2', 'h3']:
            headings = soup.find_all(tag)
            page_info["headings"].extend([h.get_text(strip=True) for h in headings])
        
        # extract links
        links = soup.find_all('a', href=True)
        page_info["links"] = [
            {"text": link.get_text(strip=True), "href": link['href']} 
            for link in links[:20]  # Limit to first 20
        ]
        
        # extract forms
        forms = soup.find_all('form')
        for form in forms:
            form_info = {
                "action": form.get('action', ''),
                "method": form.get('method', 'get'),
                "inputs": []
            }
            
            # get form inputs
            inputs = form.find_all(['input', 'select', 'textarea'])
            for inp in inputs:
                form_info["inputs"].append({
                    "type": inp.get('type', inp.name),
                    "name": inp.get('name', ''),
                    "id": inp.get('id', ''),
                    "placeholder": inp.get('placeholder', '')
                })
            
            page_info["forms"].append(form_info)
        
        # extract buttons
        buttons = soup.find_all('button')
        page_info["buttons"] = [btn.get_text(strip=True) for btn in buttons[:10]]
        
        # extract standalone inputs (not in forms)
        inputs = soup.find_all('input')
        page_info["inputs"] = [
            {
                "type": inp.get('type', 'text'),
                "name": inp.get('name', ''),
                "placeholder": inp.get('placeholder', '')
            }
            for inp in inputs[:10]
        ]
        
        # extract visible text content (simplified)
        # remove script and style elements
        for script in soup(["script", "style"]):
            script.decompose()
        
        text = soup.get_text()
        # clean up text
        lines = (line.strip() for line in text.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        text_content = ' '.join(chunk for chunk in chunks if chunk)
        
        # truncate to reasonable length for LLM
        page_info["text_content"] = text_content[:2000]
        
        await browser.close()
        
        print(f"Page analsis results")
        print(f"   - Title: {title}")
        print(f"   - Headings found: {len(page_info['headings'])}")
        print(f"   - Links found: {len(page_info['links'])}")
        print(f"   - Forms found: {len(page_info['forms'])}")
        print(f"   - Buttons found: {len(page_info['buttons'])}")
        
        return page_info

In [6]:
def format_page_info_for_llm(page_info: Dict) -> str:
    """
    Formats the page information into a structured prompt for the LLM
    """
    prompt = f"""
# Web Application Analysis

## Basic Information
- **URL**: {page_info['url']}
- **Page Title**: {page_info['title']}

## Page Structure

### Headings
{chr(10).join(f"- {h}" for h in page_info['headings'][:10])}

### Navigation Links ({len(page_info['links'])} found)
{chr(10).join(f"- {link['text']} → {link['href']}" for link in page_info['links'][:10])}

### Forms ({len(page_info['forms'])} found)
"""
    
    for i, form in enumerate(page_info['forms'], 1):
        prompt += f"\n**Form {i}:**\n"
        prompt += f"- Action: {form['action']}\n"
        prompt += f"- Method: {form['method']}\n"
        prompt += f"- Inputs:\n"
        for inp in form['inputs']:
            prompt += f"  - {inp['type']}: {inp['name']} (id: {inp['id']}, placeholder: {inp['placeholder']})\n"
    
    prompt += f"\n### Buttons\n"
    prompt += '\n'.join(f"- {btn}" for btn in page_info['buttons'][:10])
    
    prompt += f"\n\n### Page Content Preview\n"
    prompt += page_info['text_content'][:1000]
    
    return prompt

In [7]:
def generate_test_cases(page_info: Dict, model: str = "llama3.1:8b") -> List[Dict]:
    """
    Uses Ollama LLM to generate test cases based on page analysis
    """
    
    page_analysis = format_page_info_for_llm(page_info)
    
    prompt = f"""You are an expert QA automation engineer. Analyze the following web application and generate comprehensive test cases.

{page_analysis}

## Your Task
Generate 15-20 test cases for this web application. Include:
1. **Happy path tests** (things that should work normally)
2. **Edge cases** (boundary conditions, unusual inputs)
3. **Negative tests** (error handling, validation failures)
Depending on the complexity of the website, generate appropriate no of test cases so that generation is faster

For each test case, provide:
- **Test ID**: Unique identifier (e.g., TC001)
- **Test Name**: Descriptive name
- **Category**: happy_path, edge_case, or negative_test
- **Priority**: high, medium, or low
- **Description**: What the test does
- **Steps**: Detailed step-by-step actions (as array)
- **Expected Result**: What should happen

Return your response as a valid JSON array of test cases. Example format:
```json
[
  {{
    "test_id": "TC001",
    "test_name": "Successful Login with Valid Credentials",
    "category": "happy_path",
    "priority": "high",
    "description": "Verify that a user can log in successfully with valid username and password",
    "steps": [
      "Navigate to login page",
      "Enter valid username in username field",
      "Enter valid password in password field",
      "Click the login button",
      "Verify user is redirected to dashboard"
    ],
    "expected_result": "User successfully logs in and sees the dashboard"
  }}
]
```

Generate the test cases now. RETURN ONLY THE JSON ARRAY, NO OTHER TEXT.
"""
    
    # call Ollama
    response = ollama.generate(
        model=model,
        prompt=prompt,
        options={
            "temperature": 0.7,  # some creativity but not too random
            "num_predict": 4000,  # for long response
        }
    )
    
    llm_response = response['response']
    
    try:
        if "```json" in llm_response:
            json_start = llm_response.find("```json") + 7
            json_end = llm_response.find("```", json_start)
            json_str = llm_response[json_start:json_end].strip()
        elif "```" in llm_response:
            json_start = llm_response.find("```") + 3
            json_end = llm_response.find("```", json_start)
            json_str = llm_response[json_start:json_end].strip()
        else:
            json_start = llm_response.find("[")
            json_end = llm_response.rfind("]") + 1
            json_str = llm_response[json_start:json_end].strip()
        
        test_cases = json.loads(json_str)
        
        print(f"Generated {len(test_cases)} test cases")
        return test_cases
        
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON from LLM response: {e}")
        print(f"LLM Response:\n{llm_response}")
        return []

In [8]:
def display_test_cases(test_cases: List[Dict]):
    """
    Pretty print test cases
    """
    # Group by category
    categories = {}
    for tc in test_cases:
        cat = tc.get('category', 'other')
        if cat not in categories:
            categories[cat] = []
        categories[cat].append(tc)
    
    for category, tests in categories.items():
        print(f"\n{category.upper().replace('_', ' ')} ({len(tests)} tests)")
        
        for tc in tests:
            print(f"\n {tc['test_id']}: {tc['test_name']}")
            print(f"   Priority: {tc['priority'].upper()}")
            print(f"   Description: {tc['description']}")
            print(f"   Steps:")
            for i, step in enumerate(tc['steps'], 1):
                print(f"      {i}. {step}")
            print(f"   Expected: {tc['expected_result']}")

In [9]:
def save_test_cases(test_cases: List[Dict], filename: str = "test_cases.json"):
    """
    Save test cases to JSON file
    """
    with open(filename, 'w') as f:
        json.dump(test_cases, f, indent=2)
    print(f"\nTest cases saved to {filename}")

In [10]:
async def main(url: str):
    """
    Main workflow: analyze page and generate test cases
    """
    # Step 1: Analyze the page
    page_info = await analyze_page(url)
    
    # Step 2: Generate test cases
    test_cases = generate_test_cases(page_info)
    
    # Step 3: Display results
    if test_cases:
        display_test_cases(test_cases)
        save_test_cases(test_cases)
    
    return test_cases

In [17]:
url = "https://practicetestautomation.com/practice-test-login/"  

# Run the async function
test_cases = await main(url)

Page analsis results
   - Title: Test Login | Practice Test Automation
   - Headings found: 1
   - Links found: 9
   - Forms found: 0
   - Buttons found: 3
Generated 18 test cases

HAPPY PATH (2 tests)

 TC001: Successful Login with Valid Credentials
   Priority: HIGH
   Description: Verify that a user can log in successfully with valid username and password
   Steps:
      1. Navigate to login page
      2. Enter valid username 'student' in username field
      3. Enter valid password 'Password123' in password field
      4. Click the login button
      5. Verify new page URL contains 'practicetestautomation.com/logged-in-successfully/'
   Expected: User successfully logs in and sees the dashboard

 TC002: Successful Login with Valid Credentials (Alternative Path)
   Priority: HIGH
   Description: Verify that a user can log in successfully with valid username and password using an alternative path
   Steps:
      1. Navigate to login page via URL 'https://practicetestautomation.com/pr

In [3]:
import lmstudio as lms

model = lms.llm("google/gemma-3-4b")
result = model.respond("generate 5 line poem with the rhyme scheme ABABA and one rhyme word should be 'shower'")

print(result)


Okay, here’s a five-line poem with an ABABA rhyme scheme and the word “shower” included:

The rain falls soft, a gentle plea,
A welcome scent of earth so sweet, 
Reflecting skies for all to see,
Like a warm and cleansing shower complete, 
A peaceful calm, tranquility’s treat.
